# 03. ML Agent — 머신러닝 Tool 통합
**Day 14**

> Day 8~11에서 배운 머신러닝 모델을 **LangChain `@tool`** 로 감싸 Agent가 호출합니다.  
> 구현 코드: `ml_tools/` 폴더 (로지스틱 회귀 · XGBoost · One-Class SVM)

---
## 0. 설치 & 경로

In [11]:
%pip install openai python-dotenv langchain langchain-openai langchain-core langchain-classic scikit-learn xgboost joblib pandas

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ----------------- ---------------------- 3.7/8.2 MB 18.1 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.2 MB 15.5 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 14.1 MB/s  0:00:00
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   - -------------------------------------- 2.6/69.5 MB 13.7 MB/s eta 0:00:05
   --- ------------------------------------ 5.5/69.5 MB 14.0 MB/s eta 0:00:05
   ----- ---------------------------------- 8.9/69.5 MB 14.6 MB/s eta 0:00:05
   ------ --------------------------------- 11.5/69.5 MB 14.1 MB/s eta 0:00:05
   ------- -------------------------------- 13.6/69.5 MB 13.2 MB/s eta 0:00:05
   -------- ------------------------------- 14.9/69.5 MB 12.0 MB/s eta 0:00:05
   ---------- ----------------

In [12]:
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
ML_TOOLS_DIR = WORKDIR / 'ml_tools'
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

print('WORKDIR:', WORKDIR)
print('ML_TOOLS:', ML_TOOLS_DIR)
for p in sorted((ML_TOOLS_DIR / 'data').glob('*.csv')):
    print(' - data/', p.name)

WORKDIR: c:\Users\Admin\OneDrive\바탕 화면\실습\14일차_실습
ML_TOOLS: c:\Users\Admin\OneDrive\바탕 화면\실습\14일차_실습\ml_tools
 - data/ anomaly_sensor.csv
 - data/ failure_equipment.csv


---
## 1. Day 8~11 ML 방법론 요약

| Tool | 강의 | 알고리즘 | Agent가 쓰는 상황 |
|------|------|----------|-------------------|
| `predict_failure_logistic` | 로지스틱 회귀 | 설비 고장 Yes/No + **확률** (해석·설명) |
| `predict_failure_xgboost` | XGBoost | 불균형 고장 데이터 **예측 성능** 우선 |
| `detect_anomaly_oneclass` | One-Class SVM | 정상 기준 **이상 탐지** (2D 센서) |

공통 패턴:
1. `ml_tools/`에서 **사전 학습 모델** 로드 (`models/*.joblib`)
2. `@tool` 함수가 **문자열**로 결과 반환 → Agent가 읽고 답변

---
## Step 1. ML Tool 불러오기

In [13]:
from ml_tools import ML_TOOLS, predict_failure_logistic, predict_failure_xgboost, detect_anomaly_oneclass

print('등록된 ML Tool:')
for t in ML_TOOLS:
    print(f' - {t.name}: {t.description[:60]}...')

등록된 ML Tool:
 - predict_failure_logistic: 설비 고장(Failure) 여부를 로지스틱 회귀(Day 2-2)로 예측합니다.

    Args:
     ...
 - predict_failure_xgboost: 설비 고장(Failure) 여부를 XGBoost(Day 3-2)로 예측합니다.

    Args:
     ...
 - detect_anomaly_oneclass: 2차원 센서값이 정상 범위인지 One-Class SVM(Day 4-3 / SVDD)으로 탐지합니다.

   ...


---
## ✏️ 실습 1. `@tool` 직접 만들기

**목표:** 현재 시각을 `HH:MM` 형식으로 반환하는 `get_time_hhmm` Tool을 만듭니다.

**힌트:** `from langchain_core.tools import tool` · `datetime.now().strftime('%H:%M')`

In [26]:
# ── 여기에 작성 ──
from datetime import datetime
from langchain_core.tools import tool
#
@tool
def get_time_hhmm() -> str: #함수의 출력물이 어떤 형식으로 출력되는지
    """현재 시각을 HH:MM 형식으로 반환합니다."""
    return datetime.now().strftime('%H:%M')
    
print(get_time_hhmm.invoke({}))
print(type(get_time_hhmm.invoke({})))

16:49
<class 'str'>


---
## Step 2. Tool 단독 테스트

Agent 없이 `.invoke()`로 먼저 확인합니다. (14.1의 `get_current_time.invoke`와 동일)

In [15]:
# 설비 고장 예측 — 로지스틱 회귀 (Day 2-2)
print(predict_failure_logistic.invoke({
    'temperature': 67,
    'humidity': 82,
    'operator': 'Operator1',
    'hours_since_previous_failure': 90,
}))

[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 1.98%
- 정상(No) 확률: 98.02%


In [16]:
# 설비 고장 예측 — XGBoost (Day 3-2)
print(predict_failure_xgboost.invoke({
    'temperature': 67,
    'humidity': 82,
    'operator': 'Operator1',
    'hours_since_previous_failure': 90,
}))

[XGBoost] 예측: No
- 고장(Failure) 확률: 0.74%
- 정상(No) 확률: 99.26%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)


In [17]:
# 이상 탐지 — One-Class SVM (Day 4-3 / SVDD)
samples = [(1.0, 1.5, '정상에 가까운 값'), (5.0, -4.0, '이상에 가까운 값')]
for f0, f1, label in samples:
    print(f'[{label}]')
    print(detect_anomaly_oneclass.invoke({'feature_0': f0, 'feature_1': f1}))
    print('-' * 40)

[정상에 가까운 값]
[One-Class SVM] 판정: 이상
- decision_score: -0.0711
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------
[이상에 가까운 값]
[One-Class SVM] 판정: 이상
- decision_score: -0.9268
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------


---
## 센서값 직접 바꿔 `.invoke()`

`detect_anomaly_oneclass`에 임의의 `feature_0`, `feature_1` 값을 넣어 결과를 확인하세요.

**관찰:** decision_score가 0에 가까울수록 경계선 샘플입니다.

In [32]:
# ── 여기에 작성 ──
my_f0 = 1.5
my_f1 = 1.2
print(detect_anomaly_oneclass.invoke({'feature_0': my_f0, 'feature_1': my_f1}))

[One-Class SVM] 판정: 정상
- decision_score: 0.0294
- raw_prediction: 1 (+1 정상 / -1 이상)


---
## Step 3. AgentExecutor — ML Tool 통합

14.1과 같이 `create_tool_calling_agent` + `AgentExecutor`를 씁니다.

In [46]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

agent_tools = ML_TOOLS
print([t.name for t in agent_tools])

prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 제조 설비 분석을 돕는 AI입니다. 질문에 맞는 ML 도구를 골라 사용하세요.

- 설비 고장(Failure) 확률·해석 → predict_failure_logistic
- 설비 고장 예측 (XGBoost, 불균형 데이터) → predict_failure_xgboost
- 2D 센서값 정상/이상 판정 → detect_anomaly_oneclass
- 로지스틱과 XGBoost 결과 비교 → predict_failure_logistic, predict_failure_xgboost

온도·습도·작업자·이전 고장 경과시간이 주어지면 ML 예측 도구를 사용하세요.
온도,습도 중 하나라도 데이터가 없으면 결과 예측 불가로 출력하세요.
센서 feature_0, feature_1 값이 주어지면 이상 탐지 도구를 사용하세요.
도구 결과에 없으면 추측하지 마세요. 반드시 한글로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

['predict_failure_logistic', 'predict_failure_xgboost', 'detect_anomaly_oneclass']


---
## ✏️ 실습 3. 시스템 프롬프트 수정

Step 3의 `prompt` **system** 문자열에 아래 규칙을 **한 줄** 추가한 뒤, Agent를 다시 만들어 보세요.

> *"로지스틱과 XGBoost 결과를 비교해 달라고 하면 두 Tool을 모두 호출하세요."*

**힌트:** `agent` · `executor` 변수를 다시 할당하면 됩니다.

In [ ]:
# ── 여기에 작성 ──
# (Step 3 prompt 복사 후 system 문자열에 규칙 1줄 추가)
# agent = create_tool_calling_agent(llm, agent_tools, prompt)
# executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

---
## Step 4. Agent 데모 질문

In [41]:
# 로지스틱 회귀 Tool 선택 유도
r = executor.invoke({
    'input': '온도 67, 습도 82, Operator1, 이전 고장 후 90시간 — 로지스틱으로 고장 확률 알려줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 67, 'humidity': 82, 'operator': 'Operator1', 'hours_since_previous_failure': 90}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 1.98%
- 정상(No) 확률: 98.02%로지스틱 회귀 예측 결과, 고장 확률은 1.98%이며, 정상 확률은 98.02%입니다.

> Finished chain.
로지스틱 회귀 예측 결과, 고장 확률은 1.98%이며, 정상 확률은 98.02%입니다.


In [35]:
# XGBoost Tool 선택 유도
r = executor.invoke({
    'input': '같은 조건으로 XGBoost 모델 예측도 해줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_xgboost` with `{'temperature': 67, 'humidity': 82, 'operator': 'Operator1', 'hours_since_previous_failure': 5}`


[XGBoost] 예측: No
- 고장(Failure) 확률: 0.86%
- 정상(No) 확률: 99.14%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)XGBoost 모델 예측 결과는 다음과 같습니다:

- 고장(Failure) 예측: **아니오 (No)**
- 고장 확률: **0.86%**
- 정상 확률: **99.14%**

주요 변수는 다음과 같습니다:
- 온도(Temperature): 0.217
- 이전 고장 경과 시간(Hours Since Previous Failure): 0.138
- 습도(Humidity): 0.094

> Finished chain.
XGBoost 모델 예측 결과는 다음과 같습니다:

- 고장(Failure) 예측: **아니오 (No)**
- 고장 확률: **0.86%**
- 정상 확률: **99.14%**

주요 변수는 다음과 같습니다:
- 온도(Temperature): 0.217
- 이전 고장 경과 시간(Hours Since Previous Failure): 0.138
- 습도(Humidity): 0.094


In [36]:
# One-Class SVM 이상 탐지
r = executor.invoke({
    'input': '센서 feature_0=1.0, feature_1=1.5 일 때 정상인지 이상인지 판정해줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `detect_anomaly_oneclass` with `{'feature_0': 1.0, 'feature_1': 1.5}`


[One-Class SVM] 판정: 이상
- decision_score: -0.0711
- raw_prediction: -1 (+1 정상 / -1 이상)센서 feature_0=1.0, feature_1=1.5일 때 판정 결과는 "이상"입니다.

> Finished chain.
센서 feature_0=1.0, feature_1=1.5일 때 판정 결과는 "이상"입니다.


---
## ✏️ 실습 4. Agent에 본인 질문 작성

**고장 예측** 또는 **이상 탐지** 중 하나를 골라, Step 4 데모와 다른 조건으로 `executor.invoke`를 직접 호출하세요.

**예시:** *"온도 80, 습도 90, Operator3, 이전 고장 10시간 — XGBoost로 예측해줘"*

In [50]:
# ── 여기에 작성 ──
my_question = '온도 100, 습도 90, Oper 1, 이전 고장 1시간, XGBoost로 예측'
r = executor.invoke({'input': my_question, 'chat_history': []})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_xgboost` with `{'temperature': 100, 'humidity': 90, 'operator': 'Oper 1', 'hours_since_previous_failure': 1}`


[XGBoost] 예측: No
- 고장(Failure) 확률: 48.03%
- 정상(No) 확률: 51.97%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)XGBoost 예측 결과: 고장(Failure) 없음(No)  
- 고장 확률: 48.03%  
- 정상 확률: 51.97%  
- 주요 변수: 온도(Temperature) 0.217, 이전 고장 경과 시간(Hours Since Previous Failure) 0.138, 습도(Humidity) 0.094

> Finished chain.
XGBoost 예측 결과: 고장(Failure) 없음(No)  
- 고장 확률: 48.03%  
- 정상 확률: 51.97%  
- 주요 변수: 온도(Temperature) 0.217, 이전 고장 경과 시간(Hours Since Previous Failure) 0.138, 습도(Humidity) 0.094


---
## Step 5. 멀티턴 — `chat_history`

In [51]:
from langchain_core.messages import HumanMessage, AIMessage

class ChatSession:
    def __init__(self, executor):
        self.executor = executor
        self.history = []

    def ask(self, question: str) -> str:
        result = self.executor.invoke({'input': question, 'chat_history': self.history})
        answer = result['output']
        self.history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer

session = ChatSession(executor)
print('1:', session.ask('온도 70, 습도 75, Operator2, 이전 고장 120시간 — 고장 날까?'))
print('2:', session.ask('방금 답에서 고장 확률만 숫자로 다시 말해줘'))



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 70, 'humidity': 75, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 11.09%
- 정상(No) 확률: 88.91%
Invoking: `predict_failure_xgboost` with `{'temperature': 70, 'humidity': 75, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[XGBoost] 예측: Yes
- 고장(Failure) 확률: 96.66%
- 정상(No) 확률: 3.34%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)로지스틱 회귀 예측 결과:
- 고장(Failure) 확률: 11.09%
- 정상(No) 확률: 88.91%
- 예측: No (고장 발생하지 않을 것으로 예상)

XGBoost 예측 결과:
- 고장(Failure) 확률: 96.66%
- 정상(No) 확률: 3.34%
- 주요 변수(학습 기준): 온도(Temperature), 이전 고장 경과 시간(Hours Since Previous Failure), 습도(Humidity)
- 예측: Yes (고장 발생할 것으로 예상)

결과적으로, 두 모델의 예측이 상이합니다. 로지스틱 회귀는 고장이 발생하지 않을 것으로 예상하는 반면, XGBoost는 고장이 발생할 것으로 예상합니다.

> Finished chain.
1: 로지스틱 회귀 예측 결과:
- 고장(Failure) 확률: 11.09%
- 정상(No) 확률: 88.91%
- 예측: No (고장 발생하지 않을 것으로 예상

---
## ✏️ 실습 5. 멀티턴 후속 질문

`ChatSession`으로 1턴 예측 후, **2턴에서 맥락을 이어가는** 후속 질문을 직접 작성하세요.

**예시:** 1턴 고장 예측 → 2턴 *"그럼 같은 조건으로 이상 탐지도 해줘 feature_0=2, feature_1=2"*

In [53]:
# ── 여기에 작성 ──
session2 = ChatSession(executor)
q1 = '온도 70, 습도 50, Operator2, 이전 고장 120시간 — 고장 날까? XGBoost'
q2 = '그럼 같은 조건으로 이상 탐지도 해줘 feature_0=2, feature_1=2'  # 1턴 답을 참조하는 후속 질문
print('1:', session2.ask(q1))
print('2:', session2.ask(q2))



> Entering new AgentExecutor chain...

Invoking: `predict_failure_xgboost` with `{'temperature': 70, 'humidity': 50, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[XGBoost] 예측: Yes
- 고장(Failure) 확률: 99.79%
- 정상(No) 확률: 0.21%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)예측 결과, 고장(Failure) 확률은 99.79%로 매우 높습니다. 따라서 고장이 날 가능성이 큽니다.

> Finished chain.
1: 예측 결과, 고장(Failure) 확률은 99.79%로 매우 높습니다. 따라서 고장이 날 가능성이 큽니다.


> Entering new AgentExecutor chain...

Invoking: `detect_anomaly_oneclass` with `{'feature_0': 2, 'feature_1': 2}`


[One-Class SVM] 판정: 정상
- decision_score: 0.0533
- raw_prediction: 1 (+1 정상 / -1 이상)이상 탐지 결과, 판정은 정상입니다. decision_score는 0.0533으로, 정상 범위에 가깝습니다.

> Finished chain.
2: 이상 탐지 결과, 판정은 정상입니다. decision_score는 0.0533으로, 정상 범위에 가깝습니다.


---
## Step 6. (선택) 14.1 Agent와 합치기

RAG · 웹 검색 · ML을 한 Agent에 넣으려면 `agent_tools` 리스트에 합치면 됩니다.

### ✏️ 실습 6

1. **14.1 노트북에 정의**된 `get_current_time`, `search_pdf_tool`, `web_search`코드를 복사해 선언합니다.
2. 아래 셀에서 `agent_tools`를 채우고, `system` 프롬프트에 RAG·웹·ML Tool 용도를 각각 한 줄씩 추가하세요.
3. (선택) 합친 Agent로 *"석사 수업연한이 몇 학기야?"* 와 *"온도 70 습도 80 고장 예측"* 을 연속 질문해 보세요.

In [62]:
from datetime import datetime
import pytz
import yfinance as yf
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults

@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각. timezone 예: Asia/Seoul, location 예: 서울"""
    
@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """주식 가격 조회. ticker 예: TSLA, period 예: 1mo"""

web_search = DuckDuckGoSearchResults(num_results=3)

In [66]:
# ── 여기에 작성 (14.1 실행 후) ──
agent_tools = [
    get_current_time,
    get_yf_stock_history,
    web_search,
    *ML_TOOLS,
]
#
combined_prompt = ChatPromptTemplate.from_messages([
    ('system', '''... RAG / 웹 / ML Tool 용도를 각각 적으세요 ...'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])
#
combined_agent = create_tool_calling_agent(llm, agent_tools, combined_prompt)
combined_executor = AgentExecutor(agent=combined_agent, tools=agent_tools, verbose=True, max_iterations=8)

session = ChatSession(combined_executor)
print('1:',session.ask("한국vs멕시코 축구 결과"))



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_results_json` with `{'query': '한국 멕시코 축구 결과'}`


snippet: 1 week ago - 멕시코의 전략이 최근 대한민국이 보여주었던 손흥민, 황인범 등의 침투를 통한 득점을 완벽하게 차단한 셈이다. 경기 시작 전, 멕시코는 홈그라운드 및 홈팬의 응원을 등에 업고 파상공세를 펼치며 한국의 빌드업 축구를 분쇄할 거라는 전망이 많았다., title: 홍명보호(성인 2기)/2026 FIFA 월드컵/멕시코전 - 나무위키, link: https://namu.wiki/w/홍명보호(성인+2기)/2026+FIFA+월드컵/멕시코전, snippet: 3 weeks ago - 한국, 멕시코에 1대 0 패배…1승 1패, 32강 진출 미확정 ... 우리나라 축구대표팀이 북중미 월드컵에서 개최국 멕시코에 져 조별리그 1승 1패로 32강 진출을 확정하지 못했습니다., title: 한국, 멕시코에 1대 0 패배…1승 1패, 32강 진출 미확정, link: https://news.kbs.co.kr/news/view.do?ncd=8590385, snippet: 3 weeks ago - 목요일 저녁(현지 시간) 멕시코 할리스코주 과달라하라에서 열린 국제축구연맹(FIFA) 월드컵 2026 대회 A조 두 번째 경기에서 한국이 멕시코에 0-1로 패하며 1승 1패를 기록했습니다., title: 축구: 대한민국 0-1 멕시코 - FIFA 월드컵 2026, link: https://www.olympics.com/ko/news/football-korea-mexico-fifa-world-cup-2026한국 축구대표팀은 최근 멕시코와의 경기에서 0-1로 패배했습니다. 이 경기는 2026 FIFA 월드컵 A조 두 번째 경기로, 한국은 1승 1패의 성적을 기록하게 되었습니다. 32강 진출 여부는 아직 확정되지 않았습니다. 더 자세한 내용은 [여기](https://ww

In [65]:
print(session.ask('다시 말해줘'))



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_results_json` with `{'query': '한국 멕시코 축구 결과'}`


snippet: 홍명보 감독이 이끄는 남자 축구 대표팀이 9월 9일 화요일 저녁 (현지 시간) 미국 테네시주 내슈빌에서 열린 멕시코와의 친선 경기에서 2-2로 비겼습니다., title: 축구: 대한민국 2-2 멕시코 - Olympics.com, link: https://www.olympics.com/ko/news/football-korea-against-mexico-friendly-september-2025, snippet: Jun 19, 2026 · Fri, Jun 19, 2026, 01:00 UTC의 멕시코 대 한국. 실시간 결과, 상대 전적, 경기 통계, 선발 라인업, 선수 평가, 인사이트, 팀 폼, 슛맵, 하이라이트를 확인하세요., title: 멕시코 대 한국 - 실시간 점수, 예상 스타팅 라인업 및 H2H 통계, link: https://www.fotmob.com/ko/matches/mexico-vs-south-korea/1qpvav, snippet: Jun 16, 2026 · FIFA 랭킹 멕시코 13위 vs 한국 22위. 수치상 열세지만, 양팀 모두 1차전을 이기고 승점 3점 공동 선두다. 이 경기 승자가 A조 1위를 사실상 확정한다., title: 한국 멕시코 축구 친선 경기 결과 — 전적 4승 3무 8패, link: https://jiwonfund.kr/한국-멕시코-친선경기-결과/한국과 멕시코의 축구 친선 경기 결과는 2-2로 비겼습니다. 이 경기는 9월 9일 화요일 저녁에 미국 테네시주 내슈빌에서 열렸습니다. 더 자세한 내용은 [여기](https://www.olympics.com/ko/news/football-korea-against-mexico-friendly-september-2025)에서 확인하실 수 있습니다.

> Finished chai

---
## Step 7. `input()` 대화 루프

In [61]:
print('종료: exit')
while True:
    user_input = input('사용자: ').strip()
    if user_input.lower() in ('exit', 'quit', '종료'):
        break
    if not user_input:
        continue
    print('AI:', session.ask(user_input))

종료: exit


> Entering new AgentExecutor chain...
죄송하지만, 이미지 파일을 분석하거나 그 내용을 확인할 수 있는 기능은 없습니다. "Argentina.png" 파일의 사람 수를 확인하려면 해당 이미지를 직접 열어보시기 바랍니다.

> Finished chain.
AI: 죄송하지만, 이미지 파일을 분석하거나 그 내용을 확인할 수 있는 기능은 없습니다. "Argentina.png" 파일의 사람 수를 확인하려면 해당 이미지를 직접 열어보시기 바랍니다.


---
## 정리

| 파일 | 역할 |
|------|------|
| `ml_tools/logistic_regression_tool.py` | Day 9 로지스틱 회귀 |
| `ml_tools/xgboost_classifier_tool.py` | Day 10 XGBoost |
| `ml_tools/oneclass_svm_tool.py` | Day 11 One-Class SVM |
| `ml_tools/models/*.joblib` | 사전 학습 모델 (자동 생성) |

> ML Tool은 **추론만** 수행합니다. 재학습이 필요하면 각 `.py`를 `python logistic_regression_tool.py`처럼 직접 실행하세요.